# Pix2Pix — evaluación del split de prueba y demo (Colab)

Autor: Juan Sebastian Castillo Acevedo (@Juansec01)

Usa el checkpoint de la corrida base que quedó en Drive y produce, sin copiar cifras a mano:

- `summary.csv`, `per_image.csv` y `by_degradation.csv` del split **test**;
- la tabla del README / informe (`scripts/results_table.py`);
- `docs/img/results_grid.png` y `docs/img/failures.png`;
- una demo pública temporal con Gradio para tomar `docs/img/app.png`.

Ajusta las rutas de la celda 1 si tu Drive usa otras carpetas.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# Carpeta de checkpoints de la corrida base (la que imprime el notebook de entrenamiento)
CKPT_CANDIDATOS = [
    Path('/content/drive/MyDrive/pix2pix_outputs/metricas_loss_existente/checkpoints'),
    Path('/content/drive/MyDrive/Redes-Pix2Pix/resultados_pix2pix/base/checkpoints'),
]
# Checkpoint de la ablación (opcional)
ABL_CANDIDATOS = [Path('/content/drive/MyDrive/Redes-Pix2Pix/resultados_pix2pix/ablation/checkpoints')]
# Zip del dataset
ZIP_CANDIDATOS = [
    Path('/content/drive/MyDrive/Redes-Pix2Pix/datasetv1.zip'),
    Path('/content/drive/MyDrive/datasetv1.zip'),
    Path('/content/datasetv1(1).zip'),
]

def primero(rutas, obligatorio=True):
    for p in rutas:
        if p.exists():
            return p
    if obligatorio:
        raise FileNotFoundError('No se encontró ninguna de: ' + ', '.join(map(str, rutas)))
    return None

CKPT = primero(CKPT_CANDIDATOS)
ABL = primero(ABL_CANDIDATOS, obligatorio=False)
ZIP = primero(ZIP_CANDIDATOS)
print('checkpoint :', CKPT)
print('ablación   :', ABL)
print('dataset    :', ZIP)

In [ ]:
%cd /content
![ -d Trabajo-Final-de-Redes ] || git clone -q https://github.com/Isabela1105/Trabajo-Final-de-Redes.git
%cd /content/Trabajo-Final-de-Redes
!git pull -q
!pip -q install pyyaml pillow pandas matplotlib fastapi uvicorn python-multipart gradio

In [ ]:
import zipfile

REPO = Path('/content/Trabajo-Final-de-Redes')
with zipfile.ZipFile(ZIP) as z:
    tops = {n.split('/')[0] for n in z.namelist() if n.strip() and not n.startswith('__MACOSX')}
    dest = REPO if 'data' in tops else REPO / 'data' if 'processed' in tops else REPO / 'data' / 'processed'
    z.extractall(dest)

for split in ('train', 'val', 'test'):
    n = len(list((REPO / 'data/processed' / split / 'input').glob('*.png')))
    print(f'{split}: {n} pares')

## 1. Evaluar el split de prueba

In [ ]:
cmd = ['python', '-m', 'src.gan.evaluate', '--config', 'configs/base.yaml',
       '--checkpoint', str(CKPT), '--out', 'outputs/base/eval']
if ABL is not None:
    cmd += ['--model', f'U-Net solo L1={ABL}']
import subprocess
subprocess.run(cmd, check=True)

In [ ]:
import pandas as pd
display(pd.read_csv('outputs/base/eval/summary.csv'))
display(pd.read_csv('outputs/base/eval/by_degradation.csv'))

## 2. Tabla y figuras para README, informe y RESULTADOS

In [ ]:
!python scripts/results_table.py --summary outputs/base/eval/summary.csv --update README.md report/informe.md docs/RESULTADOS.md
!cp outputs/base/eval/results_grid.png outputs/base/eval/failures.png docs/img/
!git status --short

In [ ]:
from IPython.display import Image, display
display(Image('docs/img/results_grid.png'))
display(Image('docs/img/failures.png'))

## 3. Demo (para la captura `docs/img/app.png`)

Gradio crea un enlace público temporal. La demo principal (`python -m app.server`) se corre en
el portátil de la defensa, sin internet.

In [ ]:
!python -m app.gradio_app --checkpoint "$CKPT"

## 4. Descargar los archivos para subirlos al repositorio

In [ ]:
import shutil
from google.colab import files

paquete = Path('/content/resultados_evaluacion')
paquete.mkdir(exist_ok=True)
for f in ['README.md', 'report/informe.md', 'docs/RESULTADOS.md', 'docs/img/results_grid.png',
          'docs/img/failures.png', 'outputs/base/eval/summary.csv',
          'outputs/base/eval/per_image.csv', 'outputs/base/eval/by_degradation.csv']:
    destino = paquete / f
    destino.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(f, destino)
files.download(shutil.make_archive('/content/resultados_evaluacion', 'zip', paquete))